## Set Experiment Values

In [ ]:
# What is the ID for the current participant?
PARTICIPANT_ID = "jtdawc"
# How many "best" poses do you want?
NUM_POSES_WANTED = 3

# Load

In [ ]:
import numpy as np
import parse
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations

# load data
input_file = './calibration-results/{}.txt'.format(PARTICIPANT_ID)
with open(input_file, 'r') as f:
    lmk, lmk_world, trigger = parse.parse(f.read())

In [ ]:
# flatten 21 x 3 features into a vector of length 63
lmk = lmk.reshape((-1, trigger.size))

In [ ]:
# cut stream of data into trials
idx_trial_start, y, trial_length = parse.split_trial(trigger=trigger)
x_raw = np.stack([lmk[:, idx: idx + trial_length] for idx in idx_trial_start], axis=-1)

# dimensions: features (21 x 3), trial
x_raw = x_raw.mean(axis=1)
x_raw.shape

## Transform Hand Landmarks

In [ ]:
from register import HandRegister, plot_hand, reshape_to_landmark

hand_reg = HandRegister()
hand_reg.fit(x_raw)
x = hand_reg.transform(x_raw)

In [ ]:
cmap = plt.get_cmap('Set1')

fig, ax = plt.subplots(1, 2)
fig.set_size_inches(10, 5)
for ax_idx, (label, x) in enumerate([('Raw Data', x_raw),
                                     ('After Normalization', x)]):
    x = reshape_to_landmark(x)
    plt.sca(ax[ax_idx])
    for trial_idx in range(x.shape[2]):
        _x = x[:, :, trial_idx]
        # networkx insists on a 2d array
        color = np.atleast_2d(cmap(y[trial_idx]))
        plot_hand(_x[:, 0], _x[:, 1], edge_color=color)
    ax[ax_idx].title.set_text(label)

## Classification

In [ ]:
import warnings

# Suppress all future warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
x_mean_registered.T.shape

In [ ]:
z = zip(x_mean_registered.T, y)
len(list(z))

In [ ]:
# Group by trial 
label_vals = np.unique(y)

# Initialize dictionary to store separated data
separated_data = {label: [] for label in label_vals}

# Iterate over trials and separate data based on labels
for trial, label in zip(x_mean_registered.T, y):
    separated_data[label].append(trial)

# Convert lists to arrays
for label, trials in separated_data.items():
    separated_data[label] = np.array(trials).reshape(3, 3 * 21)

In [ ]:
# Set up combinations
combos = list(combinations(label_vals, NUM_POSES_WANTED))
print(len(combos))

In [ ]:
print(len(list(combos)))

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.neighbors import KNeighborsClassifier

# initialize a knn_classifier
knn_classifier = KNeighborsClassifier(n_neighbors=3)

# construction of kfold object
kfold = KFold(n_splits= NUM_POSES_WANTED, shuffle=True)

scores = np.zeros(len(combos))

for scr_idx, comb in enumerate(combos):
    num_samples = 0
    for l in comb:
        num_samples += separated_data[l].shape[0]
                
    x_classify = np.zeros((num_samples, 21*3))
    y_subset = np.zeros(num_samples)
    sample_idx = 0 
    for l in comb:
        for t in separated_data[l]:
            x_classify[sample_idx] = t
            y_subset[sample_idx] = l
            sample_idx += 1
            
    # Perform k-fold cross-validation
    cv_scores = cross_val_score(knn_classifier, x_classify, y_subset, cv=kfold)

    # Calculate the mean and standard deviation of the cross-validation scores
    scores[scr_idx] = cv_scores.mean()

    

# Find the indices of the top n scores
# top_indices = np.argsort(scores)[::-1][:3]
# top_combos = []
# top_scores = []

# for idx in top_indices:
#     top_combos.append(combos[idx])
#     top_scores.append(scores[idx])
    
# print(top_scores)
# print(top_combos)

# Find the top score
highest_score_idx = np.argmax(scores)
highest_score = scores[highest_score_idx]
best_combo = list(combos[highest_score_idx])
print(highest_score)
print(best_combo)

## Ordered Hand Symbols

In [ ]:
def find_best_combo(remaining_labels):
    # Set up combinations
    combos = list(combinations(remaining_labels, len(remaining_labels) - 1))
    
    # initialize a knn_classifier
    knn_classifier = KNeighborsClassifier(n_neighbors=3)

    # construction of kfold object
    kfold = KFold(n_splits= NUM_POSES_WANTED, shuffle=True)

    scores = np.zeros(len(combos))

    for scr_idx, comb in enumerate(combos):
        num_samples = 0
        for l in comb:
            num_samples += separated_data[l].shape[0]

        x_classify = np.zeros((num_samples, 21*3))
        y_subset = np.zeros(num_samples)
        sample_idx = 0 
        for l in comb:
            for t in separated_data[l]:
                x_classify[sample_idx] = t
                y_subset[sample_idx] = l
                sample_idx += 1

        # Perform k-fold cross-validation
        cv_scores = cross_val_score(knn_classifier, x_classify, y_subset, cv=kfold)

        # Calculate the mean and standard deviation of the cross-validation scores
        scores[scr_idx] = cv_scores.mean()

    # Find the top score
    highest_score_idx = np.argmax(scores)
    highest_score = scores[highest_score_idx]
    best_combo = list(combos[highest_score_idx])
    
    return best_combo

In [ ]:
ordered_poses = np.empty(len(label_vals))
remaining_poses = np.copy(label_vals)

for n in range(len(label_vals) - 1, 1, -1):
    best = find_best_combo(remaining_poses)
    least_distinguished = np.setdiff1d(remaining_poses, best)[0]
    print('{}: OF: {}, BEST: {}, WORST: {}'.format(n, remaining_poses, best, least_distinguished))
    remaining_poses = np.delete(remaining_poses, np.where(remaining_poses == least_distinguished))
    ordered_poses[n] = least_distinguished
    if n == 2:
        ordered_poses[0] = best[0]
        ordered_poses[1] = best[1]

print(ordered_poses)